# Bankruptcy Prediction
Predict company bankruptcy from financial ratios.

## Project Overview
Binary classification on the Taiwan Economic Journal company bankruptcy dataset. 95 financial ratio features, ~6,800 companies.

## Learning Objectives
- Handle high-dimensional financial data (95 features)
- Work with extreme class imbalance (~3% bankruptcy)
- Feature importance analysis for financial risk

## Problem Statement
Given 95 financial ratios of a company, predict whether it will go bankrupt.

## Why This Project Matters
Bankruptcy prediction is central to credit risk modeling, investment analysis, and regulatory compliance.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: fedesoriano/company-bankruptcy-prediction |
| **Target** | Bankrupt? (0/1) |
| **Rows** | ~6,800 |
| **Features** | 95 financial ratios |

## Dataset Source & License
Taiwan Economic Journal 1999-2009. Kaggle: CC0 Public Domain.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
MAX_ROWS = 50000

## Dataset Loading

In [4]:
import kagglehub
path = kagglehub.dataset_download('fedesoriano/company-bankruptcy-prediction')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', [os.path.basename(f) for f in csvs])
df = pd.read_csv(csvs[0])
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

  0%|                                              | 0.00/4.63M [00:00<?, ?B/s]

 22%|████████▏                             | 1.00M/4.63M [00:01<00:05, 668kB/s]

 43%|███████████████▉                     | 2.00M/4.63M [00:01<00:02, 1.34MB/s]

 65%|███████████████████████▉             | 3.00M/4.63M [00:01<00:00, 2.15MB/s]

100%|█████████████████████████████████████| 4.63M/4.63M [00:02<00:00, 3.73MB/s]

100%|█████████████████████████████████████| 4.63M/4.63M [00:02<00:00, 2.33MB/s]

Extracting files...


Files: ['data.csv']
Shape: (6819, 96)


,Bankrupt?,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,1,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,...,0.716845,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469
1,1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,...,0.795297,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794
2,1,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,...,0.774670,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474
3,1,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,...,0.739555,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982
4,1,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,...,0.795016,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490


## Data Validation

In [5]:
print('Missing:', df.isnull().sum().sum())
print(f'Duplicates: {df.duplicated().sum()}')

# Find target
bankrupt_cols = [c for c in df.columns if 'bankrupt' in c.lower()]
TARGET = bankrupt_cols[0] if bankrupt_cols else df.columns[0]
print(f'\nTarget: {TARGET}')
print(df[TARGET].value_counts())
print(f'Bankruptcy rate: {df[TARGET].mean():.2%}')

Missing: 0


Duplicates: 0

Target: Bankrupt?
Bankrupt?
0    6599
1     220
Name: count, dtype: int64
Bankruptcy rate: 3.23%


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#2ecc71','#e74c3c'], edgecolor='black')
axes[0,0].set_title('Bankruptcy Distribution')

num_cols = [c for c in df.select_dtypes('number').columns if c != TARGET]
# Top correlated features
corr = df[num_cols].corrwith(df[TARGET]).abs().sort_values(ascending=False)
corr.head(10).plot.bar(ax=axes[0,1], edgecolor='black')
axes[0,1].set_title('Top Correlated Features')
axes[0,1].tick_params(axis='x', rotation=45, labelsize=7)

# Feature distributions for top 2
for i, col in enumerate(corr.head(2).index):
    ax = axes[1, i]
    for label in [0, 1]:
        subset = df[df[TARGET] == label]
        ax.hist(subset[col].clip(lower=subset[col].quantile(0.01), upper=subset[col].quantile(0.99)),
                bins=30, alpha=0.5, label=f'{"Bankrupt" if label else "Healthy"}')
    ax.legend(fontsize=8)
    ax.set_title(col[:40])
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].value_counts())
print(f'\nImbalance ratio: {(df[TARGET]==0).sum() / max((df[TARGET]==1).sum(), 1):.0f}:1')

Bankrupt?
0    6599
1     220
Name: count, dtype: int64

Imbalance ratio: 30:1


## Train/Test Split

In [8]:
# Clean data
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(df.median(numeric_only=True))

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train bankruptcy rate: {y_train.mean():.2%}')

Train: (5455, 95), Test: (1364, 95)
Train bankruptcy rate: 3.23%


## Preprocessing
Replaced inf with NaN, filled with median. All features are numeric financial ratios.

## Feature Engineering
Using raw 95 financial ratios. Tree models handle feature interactions and selection internally.

## Baseline Model

In [9]:
bl = LogisticRegression(max_iter=1000, random_state=SEED, class_weight='balanced')
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR (balanced): Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}  Recall={recall_score(y_test, bl_pred):.4f}')

Baseline LR (balanced): Acc=0.7478  F1=0.0851  Recall=0.3636


## LazyPredict Benchmark

In [10]:
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  \
Model                                                                         
BernoulliNB                    0.807           0.862051  0.925826  0.872955   
NearestCentroid                0.895           0.829231  0.927015  0.926514   
Perceptron                     0.941           0.755385  0.822523  0.952975   
DecisionTreeClassifier         0.967           0.690769  0.690769  0.967909   
LGBMClassifier                 0.977           0.676410  0.950113  0.974528   
XGBClassifier                  0.976           0.656410  0.931077  0.973061   
LinearDiscriminantAnalysis     0.959           0.647692  0.910482  0.961444   
KNeighborsClassifier           0.976           0.617436  0.726831  0.971419   
RidgeClassifier                0.976           0.597949  0.912492  0.970451   
CatBoostClassifier             0.974           0.596923  0.937641  0.969037   
BaggingClassifier              0.973           0.596

## FLAML AutoML

In [11]:
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Acc={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
models = {}
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

xgb_m = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_m.fit(X_train, y_train)
models['XGBoost'] = xgb_m

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

CatBoost: Acc=0.9729  F1=0.9676
LightGBM: Acc=0.9721  F1=0.9651
XGBoost: Acc=0.9729  F1=0.9687


## Top 3 Model Selection

In [13]:
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

          Accuracy        F1  Precision    Recall
XGBoost   0.972874  0.968730   0.967852  0.972874
CatBoost  0.972874  0.967619   0.967508  0.972874
LightGBM  0.972141  0.965092   0.966411  0.972141

Top 3: ['XGBoost', 'CatBoost', 'LightGBM']


## Final Evaluation of Top 3

In [14]:
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

XGBoost: Acc=0.9729  F1=0.9687
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1320
           1       0.65      0.34      0.45        44

    accuracy                           0.97      1364
   macro avg       0.82      0.67      0.72      1364
weighted avg       0.97      0.97      0.97      1364

CatBoost: Acc=0.9729  F1=0.9676
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      1320
           1       0.68      0.30      0.41        44

    accuracy                           0.97      1364
   macro avg       0.83      0.65      0.70      1364
weighted avg       0.97      0.97      0.97      1364

LightGBM: Acc=0.9721  F1=0.9651
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      1320
           1       0.71      0.23      0.34        44

    accuracy                           0.97      1364
   macro avg       0.84      0.61

## Error Analysis

In [15]:
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

err = (preds != y_test.values).sum()
print(f'Errors: {err}, Error rate: {err/len(y_test):.4f}')

Errors: 37, Error rate: 0.0271


## Interpretation & Business Insight
Debt ratio, ROA, and net income features are key bankruptcy indicators. Early warning systems using these ratios can prevent bad loans.

## Limitations
- Extreme imbalance (~3% bankrupt)
- Taiwan-specific data may not generalize
- Financial ratios may be manipulated

## How to Improve
- SMOTE or class weights for imbalance
- Feature selection to reduce 95 → top 20
- Time-based validation

## Production Considerations
- Quarterly model updates with new filings
- Explainable predictions for regulators
- Integration with credit scoring systems

## Common Mistakes
- Ignoring class imbalance
- Not handling inf values in financial ratios
- Using accuracy instead of recall/PR-AUC

## Mini Challenge
1. Select top 20 features and compare
2. Try SMOTE oversampling
3. Plot precision-recall curve

## Final Summary
Predicted company bankruptcy from 95 financial ratios. Handling extreme imbalance is key — recall and F1 matter more than accuracy.

In [16]:
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "XGBoost": {
    "Accuracy": 0.9729,
    "F1": 0.9687,
    "Precision": 0.9679,
    "Recall": 0.9729
  },
  "CatBoost": {
    "Accuracy": 0.9729,
    "F1": 0.9676,
    "Precision": 0.9675,
    "Recall": 0.9729
  },
  "LightGBM": {
    "Accuracy": 0.9721,
    "F1": 0.9651,
    "Precision": 0.9664,
    "Recall": 0.9721
  },
  "best_model": "XGBoost"
}
